# 07. AI 트렌드 보고서 기반 출처 포함 답변 생성

## 실습 목적

이번 실습에서는 앞에서 만든 **AI 트렌드 보고서 Markdown 기반 Vector Store**를 사용한다.

목표는 단순히 답변을 생성하는 것이 아니라, **어떤 문서를 근거로 답변했는지 함께 확인하는 RAG 체인**을 만드는 것이다.

## 왜 출처가 중요한가?

- 사용자가 답변의 근거 문서를 확인할 수 있다.
- LLM이 문서에 없는 내용을 임의로 말하는 문제를 줄일 수 있다.
- 검색된 문서가 실제 질문과 관련 있는지 검토할 수 있다.

## 이번 실습에서 다루는 흐름

```text
질문 입력
  ↓
Vector Store에서 관련 문서 검색
  ↓
검색 문서에 번호 붙이기
  ↓
question + context를 프롬프트에 입력
  ↓
LLM 답변 생성
  ↓
답변과 참고 문서 함께 출력
```

> 이 노트북은 `02_data_pdf_to_md.ipynb`에서 만든 ChromaDB를 사용한다.


## 검색 문서 출처 정보 정리 함수

In [1]:
def doc_label(doc):
    """검색 문서의 출처 정보를 보기 좋게 정리한다."""
    metadata = doc.metadata

    source = metadata.get("source", "unknown")
    chunk_id = metadata.get("chunk_id", "?")
    doc_type = metadata.get("doc_type", "?")
    source_name = metadata.get("source_name", "?")

    return f"chunk_id={chunk_id} | doc_type={doc_type} | source_name={source_name} | source={source}"

In [ ]:
from dotenv import load_dotenv
from pathlib import Path
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

# --------------------------------------------------
# 1. 환경 변수 로딩
# --------------------------------------------------

load_dotenv(override=True, dotenv_path="../.env")

# --------------------------------------------------
# 2. LLM과 Embedding 모델 준비
# --------------------------------------------------

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# --------------------------------------------------
# 3. 앞 실습에서 만든 AI 트렌드 보고서 Vector Store 불러오기
# --------------------------------------------------

DB_PATH = Path("chroma_db/ai_trend_report_md")
COLLECTION_NAME = "ai_trend_report_2026_md"

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=DB_PATH
)

retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

print("AI 트렌드 보고서 Vector Store 준비 완료")
print(f"저장된 청크 수: {vector_store._collection.count()}")


AI 트렌드 보고서 Vector Store 준비 완료
저장된 청크 수: 29


## 먼저 검색 결과를 확인하기

출처 기반 답변을 만들기 전에, 질문을 넣었을 때 어떤 문서 조각이 검색되는지 먼저 확인한다.

검색 결과가 적절하지 않으면 답변도 정확하기 어렵다.


In [3]:
question = "AI 산업, AI 기술, AI 정책의 핵심 이슈를 비교해줘."

docs = retriever.invoke(question)

print(f"검색된 문서 수: {len(docs)}")

for i, doc in enumerate(docs, start=1):
    print(f"\n[문서 {i}]")
    print(doc_label(doc))
    print(doc.page_content[:500])
    print("-" * 80)


검색된 문서 수: 4

[문서 1]
chunk_id=6 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.md
- - 산업·정책 영역에서는 제도화 지연에 따른 불확실성과 대응 부담이 증가하고 있으며, 국가·기업 모두 AI 활용과 규제 준수 사이에서 복합적 리스크에 직면하고 있음 

> 1) Stanford HAI, 2025 AI Index Report – Economy 

> 2) OECD AI Incidents Monitor(AIM) 

## **1.2 분석 목적** 

- 본 이슈페이퍼는 2025년을 전후해 급격히 재편되는 글로벌 AI 재편 흐름을 진단 하기 위해, 산업·기술·정책 전반에 반복적으로 등장하는 핵심 키워드와 국제 담론을 종합적으로 수집·분석하는 것을 목적으로 함 

- - 2025년을 분석 기준 시점으로 설정함으로써, 2023~2024년의 실험·도입 국면에서 2025년 이후 전면 확산·제도화 국면으로의 전환이 어떻게 나타나는지를 계량적으로 비교·검증하고자 함 

- - 단순 기사 검토 수준을 넘어 글로벌 싱크탱크·정책 보고서·산업 리포트 등 다층적 자료를 기반으로 하여 AI
--------------------------------------------------------------------------------

[문서 2]
chunk_id=19 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.md
## **3.2.4 [AI 산업·기술·정책 토픽] - 종합 결과 및 시사점** 

## **< 표 4 > AI 산업·기술·정책 토픽 모델링 종합 결과 및 시사점** 

|**분야**|**토픽명**|**주요 키워드**|**핵심 이슈**|
|---|---|---|---|
|AI 산업|시장 규모
확대와 

## 검색 문서에 번호를 붙여 context 만들기

LLM에게 검색 문서를 그냥 전달하면 어떤 문서를 근거로 답했는지 알기 어렵다.

따라서 검색된 문서에 `[문서 1]`, `[문서 2]`처럼 번호를 붙여 전달한다.

이렇게 하면 답변에서도 근거 문서 번호를 표시하도록 유도할 수 있다.


In [4]:
def format_docs_with_source(docs):
    """검색된 문서에 번호와 metadata를 붙여 하나의 context 문자열로 변환한다."""

    formatted_docs = []

    for i, doc in enumerate(docs, start=1):
        metadata = doc.metadata

        source = metadata.get("source", "unknown")
        chunk_id = metadata.get("chunk_id", "?")
        doc_type = metadata.get("doc_type", "?")
        source_name = metadata.get("source_name", "?")

        formatted_doc = f"""[문서 {i}]
source: {source}
source_name: {source_name}
chunk_id: {chunk_id}
doc_type: {doc_type}

{doc.page_content}"""

        formatted_docs.append(formatted_doc)

    return "\n\n".join(formatted_docs)

## 출처 번호를 포함해 답변 생성하기

아래 프롬프트는 LLM에게 세 가지 규칙을 준다.

1. 검색된 문서에 근거해서만 답한다.
2. 문서에서 확인할 수 없는 내용은 확인할 수 없다고 답한다.
3. 답변 문장 끝에 `[문서 1]`처럼 근거 번호를 표시한다.


In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

source_answer_prompt = ChatPromptTemplate.from_messages([
    ("system", """아래 문서에 근거해서만 질문에 답하세요.
문서에서 확인할 수 없는 내용은 '문서에서 확인할 수 없습니다'라고 답하세요.
답변에 문서 내용을 사용할 때는 문장 끝에 [문서 1], [문서 2] 형식으로 근거를 표시하세요.
한 줄에 한 문장씩 깔끔하게 정리하세요.

[문서]
{context}"""),
    ("human", "{question}")
])

source_answer_chain = source_answer_prompt | llm | StrOutputParser()

question = "AI 산업, AI 기술, AI 정책의 핵심 이슈를 비교해줘."

docs = vector_store.similarity_search(question, k=4)
context = format_docs_with_source(docs)

# 프롬프트에 question과 context를 넣는다.
answer = source_answer_chain.invoke({
    "question": question,
    "context": context
})
print("[출처 포함 답변]")
print(answer)


[출처 포함 답변]
AI 산업의 핵심 이슈는 산업 구조 변화와 글로벌 경쟁 심화입니다. 

AI 기술의 핵심 이슈는 지능 구조 고도화와 기술 적용 환경의 다변화입니다. 

AI 정책의 핵심 이슈는 안전성 확보와 위험 관리 중심의 정책 및 법적 틀 구축입니다. 

세 분야는 서로 다른 변화 축을 가지지만, 2025년을 전후한 복합적 전환기에 진입하고 있습니다 [문서 2].


## 답변과 참고 문서를 함께 출력하는 함수 만들기

매번 같은 코드를 반복하지 않도록 간단한 함수로 묶는다.

이 함수는 다음 순서로 동작한다.

```text
질문 입력
  ↓
관련 문서 검색
  ↓
검색 문서에 번호 붙이기
  ↓
LLM 답변 생성
  ↓
답변과 참고 문서 출력
```


In [ ]:
def answer_with_sources(question, k=4):
    """질문에 답하고, 사용한 검색 문서를 함께 출력한 뒤 결과를 반환한다."""

    # # 1. 관련 문서 검색
    # docs = vector_store.similarity_search(question, k=k)

    # # 2. 검색 문서를 context로 변환
    # context = format_docs_with_source(docs)

    # # 3. 프롬프트 생성
    # messages = source_answer_prompt.invoke({
    #     "question": question,
    #     "context": context
    # })

    # # 4. LLM 답변 생성
    # response = llm.invoke(messages)
    # answer = StrOutputParser().invoke(response)

    # LCEL 체인으로 간소화
    source_answer_chain = source_answer_prompt | llm | StrOutputParser()
    docs = vector_store.similarity_search(question, k=k)
    context = format_docs_with_source(docs)
    answer = source_answer_chain.invoke({
        "question": question,
        "context": context
    })

    # 5. 답변 출력
    print("[질문]")
    print(question)

    print("\n[답변]")
    print(answer)

    print("\n[참고 문서]")
    for i, doc in enumerate(docs, start=1):
        preview = doc.page_content[:150]
        print(f"{i}. {doc_label(doc)}")
        print(f"   {preview}...")

    return {
        "question": question,
        "answer": answer,
        "docs": docs,
        "context": context
    }


In [7]:
answer_with_sources(
    "2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?",
    k=4
)


[질문]
2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?

[답변]
2026년 AI 기술 전망에서 핵심 변화는 고품질 데이터 생성, 멀티모달 및 고급 추론 기술의 결합으로 AI의 상황 이해와 문제 해결 능력이 향상되는 것입니다. 

또한, 기술 고도화에 따라 안전성 및 정합성 검증의 중요성이 커지며, 신뢰성 평가와 표준화가 AI 기술 확산의 기반 인프라로 자리 잡을 것으로 전망됩니다. 

마지막으로, 데이터 품질 표준화와 모델 신뢰성 평가 기준 구축이 기능 고도화 기술의 산업 적용 확산을 뒷받침할 핵심 인프라로 요구될 것입니다[문서 3].

[참고 문서]
1. chunk_id=21 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.md
   ## **4.1 2025년 AI 산업·기술·정책의 트렌드 현황** 

- 앞 장에서는 2025년 주요 매체 데이터를 바탕으로 산업·기술·정책 전반에서 나타난 AI 담론의 구조적 흐름을 도출하였음 

- - 분석 결과를 통해 세 분야는 서로 다른 영역에서 출발하더라도 활...
2. chunk_id=6 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.md
   - - 산업·정책 영역에서는 제도화 지연에 따른 불확실성과 대응 부담이 증가하고 있으며, 국가·기업 모두 AI 활용과 규제 준수 사이에서 복합적 리스크에 직면하고 있음 

> 1) Stanford HAI, 2025 AI Index Report – Economy 

> ...
3. chunk_id=23 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.md
   - - 고품질 데이터 생성·멀티모달·고

{'question': '2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?',
 'answer': '2026년 AI 기술 전망에서 핵심 변화는 고품질 데이터 생성, 멀티모달 및 고급 추론 기술의 결합으로 AI의 상황 이해와 문제 해결 능력이 향상되는 것입니다. \n\n또한, 기술 고도화에 따라 안전성 및 정합성 검증의 중요성이 커지며, 신뢰성 평가와 표준화가 AI 기술 확산의 기반 인프라로 자리 잡을 것으로 전망됩니다. \n\n마지막으로, 데이터 품질 표준화와 모델 신뢰성 평가 기준 구축이 기능 고도화 기술의 산업 적용 확산을 뒷받침할 핵심 인프라로 요구될 것입니다[문서 3].',
 'docs': [Document(id='e7f208d0-5e94-4f56-8992-d43b3c0e7bb8', metadata={'chunk_id': 21, 'doc_type': 'markdown', 'source': 'data\\AI@Data_Report_CLEANED.md', 'source_name': 'AI@Data_Report_CLEANED.md'}, page_content='## **4.1 2025년 AI 산업·기술·정책의 트렌드 현황** \n\n- 앞 장에서는 2025년 주요 매체 데이터를 바탕으로 산업·기술·정책 전반에서 나타난 AI 담론의 구조적 흐름을 도출하였음 \n\n- - 분석 결과를 통해 세 분야는 서로 다른 영역에서 출발하더라도 활용 확대 → 기술적 고도화 요구 증가 → 제도적 대응 심화 로 이어지는 연속적 흐름을 형성하고 있음을 확인하였음 \n\n- 2026년에는 이 세 방향성이 더욱 맞물려 산업 구조의 재배치, 기술 체계의 정교화, 규제·책임 체계의 본격적 적용으로 이어지는 개략적 전망을 제시함 \n\n## **4.2 2026년 AI 산업 전망** \n\n2026년 AI 시장은 선도 기업을 중심으로 내재화가 빠르게 진행 되며 시장 규모가 지속 확대될 전망 \n\n- - AI 활용이 여러 산업군에서 실험 단계를 넘어 일부 핵심 공정과 서비스 운영에 본격

## 문서에 없는 질문으로 확인하기

출처 기반 RAG에서는 문서에 없는 내용을 억지로 답하지 않도록 만드는 것이 중요하다.

아래 질문은 AI 트렌드 보고서에 직접적인 근거가 없을 가능성이 높은 질문이다.

이 경우 답변은 일반 지식으로 확장하지 않고, 문서에서 확인할 수 없다고 말해야 한다.


In [8]:
answer_with_sources(
    "이 보고서에서 GPT-5의 성능 평가 결과는 어떻게 제시되었나요?",
    k=4
)


[질문]
이 보고서에서 GPT-5의 성능 평가 결과는 어떻게 제시되었나요?

[답변]
문서에서 확인할 수 없습니다.

[참고 문서]
1. chunk_id=26 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.md
   ## **5.2 AI 기술 토픽으로 본 노력 및 대응** 

- 추론(AI Reasoning)의 중요성이 커짐에 따라 정부에서는 AI 확산 생태계 조성을 위하여 추론형 AI 모델 개발에 필수적인 고품질 추론 데이터 구축 및 활용 체계를 선제적으로 정비 

- - 고품질...
2. chunk_id=1 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.md
   AI@Data Report **
-제3호** 

**목차** 

❶ **배경 및 목적 /p.06** 

❷ **자료 분석 및 방법 /p.09** 

- ❸ **분석 결과 /p.12** 

- ❹ **2026년 전망: AI 생태계 방향성 /p.17** 

❺ **분석 결...
3. chunk_id=16 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.md
   - 현시점 AI 산업은 시장 규모 확대, 도입 가속, 인프라 중심 운영 모델화가 동시에 나타나며, AI가 산업의 핵심 인프라로 자리 잡아가는 흐름 을 시사함 

## **3.2.2 [AI 기술 토픽] - 기능 고도화와 적용 범위 확장** 

## **[ 그림 3 ] 기...
4. chunk_id=8 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.

{'question': '이 보고서에서 GPT-5의 성능 평가 결과는 어떻게 제시되었나요?',
 'answer': '문서에서 확인할 수 없습니다.',
 'docs': [Document(id='4697ffa9-f0a3-458b-bd12-0bb39fb2841a', metadata={'chunk_id': 26, 'source': 'data\\AI@Data_Report_CLEANED.md', 'source_name': 'AI@Data_Report_CLEANED.md', 'doc_type': 'markdown'}, page_content='## **5.2 AI 기술 토픽으로 본 노력 및 대응** \n\n- 추론(AI Reasoning)의 중요성이 커짐에 따라 정부에서는 AI 확산 생태계 조성을 위하여 추론형 AI 모델 개발에 필수적인 고품질 추론 데이터 구축 및 활용 체계를 선제적으로 정비 \n\n- - 고품질·고난이도 문제 해결 중심의 추론 데이터를 기획·개발하여 기존의 단순 인식·예측 중심 데이터에서 벗어나, ‘단계별 과정·의사결정 논리·맥락’을 포함한 구조화된 추론형 데이터셋 구축 진행 \n\n## **< 추론형 데이터 종류(예시) >**'),
  Document(id='fedcaa02-98c8-4301-8485-79a4bed1a8f4', metadata={'source_name': 'AI@Data_Report_CLEANED.md', 'source': 'data\\AI@Data_Report_CLEANED.md', 'doc_type': 'markdown', 'chunk_id': 1}, page_content='AI@Data Report **\n-제3호** \n\n**목차** \n\n❶ **배경 및 목적 /p.06** \n\n❷ **자료 분석 및 방법 /p.09** \n\n- ❸ **분석 결과 /p.12** \n\n- ❹ **2026년 전망: AI 생태계 방향성 /p.17** \n\n❺ **분석 결과로 본 우리의 대응 및 노력 /p.20** \n\n## AI@Data

## 정리

| 단계 | 내용 |
|---|---|
| 1 | 질문으로 관련 문서를 검색한다. |
| 2 | 검색된 문서에 `[문서 1]`, `[문서 2]`처럼 번호를 붙인다. |
| 3 | 번호가 붙은 문서를 context로 만들어 LLM에 전달한다. |
| 4 | LLM은 context에 근거해서 답변한다. |
| 5 | 답변과 함께 참고 문서를 출력한다. |

## 핵심 포인트

- 출처 기반 답변은 RAG의 신뢰도를 높이는 기본 기법이다.
- Markdown 기반 ChromaDB는 페이지 번호가 없을 수 있으므로, 이번 실습에서는 문서 번호와 source 정보를 출처로 사용한다.
- `[문서 1]`, `[문서 2]`는 원문 보고서의 고정 문서 번호가 아니라, 현재 질문에 대해 검색된 청크의 순서 번호이다.
- 문서에 없는 내용은 답하지 않도록 프롬프트에서 명확히 제한해야 한다.
- 실제 서비스에서는 문서 번호뿐 아니라 파일명, 페이지, 섹션 제목, URL 등을 함께 저장하면 더 좋다.

다음 단계에서는 RAG 답변의 품질을 평가하는 방법을 다룬다.
